# CS-781 Phase 5 (Legal Statute Identification) - `p5_moe_lorma`

**Phase-1 model:** MoE (Muon + final LayerNorm), continuation 2 on high-court data (218.7M params).  **This notebook:** Round 2, all training cases, 16 x 6,000 cases: the MoE with LoRMA (same budget as moe_partial2_long: LoRMA against partial fine-tuning on the same model).

One Run All trains every setup in `SWEEP` (cell 1) on ALL unique training cases (`FULL_RUN = True`). The model reads the case in chunks of 512 tokens (sliding window, at most 8 chunks: first 2 + last 6 for long cases), pools each chunk (mean + last token), and a multi-label document head (BiLSTM + attention over the chunks) gives one score per statute (100 statutes); the loss is binary cross-entropy with inverse-sqrt positive weights. After each "epoch" (6,000 cases) a fixed sample of 2,000 selection-set cases is scored (macro-F1 at the best global threshold) and the best epoch is kept. At the end the full dev set and the test set are scored once, and thresholds are fitted: one global threshold, one per statute, and a shrunk mixture of both; their honest value is estimated by fitting on one half of the selection set and scoring the other half. The selection set is the dev set without the 67 dev cases whose text also occurs in test. Every finished setup is cached in the bucket (resumable). Finally it prints the comparison table and writes one `submission_*.zip` per full-data run (thresholds of the mode with the best honest score).

**Kaggle:** GPU T4, Internet on, secrets `HF` and `HF_WRITE`. Set `QUICK_TEST = True` in cell 1 first for a few-minute check of the whole pipeline. Run `p5_tokenize` once first (CPU) so that the tokenised cases are in the bucket.

All hyper-parameter values are placeholders proposed by the assistant (state this in your report).

## 1. Settings

In [1]:
# [P5-PRELUDE:settings]
import json
# =====================================================================================================
# SETTINGS -- every value is a placeholder you can change. Hyper-parameter values were proposed by the assistant and are NOT tuned.
# =====================================================================================================
SWEEP_NAME = "p5_moe_lorma"
MODEL_KEY = "moe_hc"
PHASE1_CHECKPOINT = "phase1_decoder_only_gqa_kvcache_moe_muon_finalnorm_continued2_highcourt_fp16_full_checkpoint.pt"
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
PEFT_TARGETS = ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj"]
IA3_OUT = ["attn.k_proj", "attn.v_proj"]
IA3_IN = ["ff.2", ".down"]
MOE_BIAS_RATE_FT = 0.0       # MoE routing-bias update rate during fine-tuning (Phase 1 used 0.001). 0.0 = keep the pretrained routing fixed.
ATTENTION = "causal"   # only used by the Phase-1 decoder code below

# ---- how much data: the main notebooks train on ALL training cases; the ablation notebooks use a SUBSET (same subset everywhere: a fixed shuffle, first N) ----
FULL_RUN = True               # True = all training cases; False = the first MAX_TRAIN_DOCS of a fixed shuffle (used for the ablations)
MAX_TRAIN_DOCS = 6000      # used when FULL_RUN is False
MAX_DEV_DOCS = None            # None = the whole dev set (it is the model-selection set)
MAX_DOC_TOKENS = 16384         # a longer case keeps its first and last half of this many tokens (99% of cases are shorter)

BASE = dict(
    N_LABELS=None,           # filled from the data (100 statutes, ids 0..99)
    TECHNIQUE="lorma", RANK=16, ALPHA=32, LORA_DROPOUT=0.0, PEFT_TARGETS=PEFT_TARGETS, LORMA_VARIANT="plus",
    ADALORA_INIT_RANK=24, ADALORA_TARGET_RANK=8, ADALORA_DELTA_T=10, ADALORA_ORTH=0.1,
    IA3_OUT=IA3_OUT, IA3_IN=IA3_IN, ADAPTER_KIND="houlsby", ADAPTER_R=64, PARTIAL_LAYERS=2,
    # long-document representation: chunks of WINDOW tokens, stride STRIDE (< WINDOW = overlapping), at most MAX_CHUNKS kept (SELECT chooses which)
    WINDOW=512, STRIDE=512, MAX_CHUNKS=8, SELECT="head_tail", HEAD_CHUNKS=2,
    POOL="mean+last", DOC_HEAD="bilstm", HEAD_LSTM_HIDDEN=256, HEAD_DROPOUT=0.1, HEAD_NORM=True, LABEL_MIX=False,
    # loss: bce | bce_pw | focal | asl ; pos_weight_l = ((N - n_l) / n_l) ** PW_POWER capped at PW_CAP (then divided by its mean)
    LOSS="bce_pw", PW_POWER=0.5, PW_CAP=10.0, FOCAL_GAMMA=2.0, ASL_GAMMA_POS=0.0, ASL_GAMMA_NEG=4.0, ASL_CLIP=0.05, LOSS_SCALE=10.0, LABEL_SMOOTHING=0.0,
    # training recipe (an "epoch" = EPOCH_DOCS documents drawn from the training set)
    BATCH=1, ACCUM=16, ROW_GROUP=8, CKPT=True, INFER_BATCH=4, EPOCHS=16, EPOCH_DOCS=6000, DEV_SUB=2000, PATIENCE=0,
    LR=3e-4, HEAD_LR=1e-3, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.06, GRAD_CLIP=1.0, AMP="fp16", SEED=0,
    SELECT_METRIC="f1_tuned", TIME_BUDGET_MIN=None,
    KD_ALPHA=0.0, KD_TEMPERATURE=2.0, KD_TEACHER=None, SAVE_TRAIN_LOGITS=False,
    SAVE_WEIGHTS=False)      # True = also upload the trained (adapter / head / unfrozen) parameters of the best epoch to the bucket as p5w_<notebook>__<setup>.pt

LR_BY_TECH = dict(full_ft=3e-5, partial=1e-4, bitfit=1e-3, adapters=5e-4, frozen=3e-4)   # learning rate of the backbone parameters per technique

SWEEP = [   # (name, settings that differ from BASE). Order = run order.
    ("moe_lorma_long", dict(TECHNIQUE='lorma', SAVE_WEIGHTS=True)),
]

# ---- KD: a setup with KD_TEACHER waits up to TEACHER_WAIT_MIN minutes for the teacher's result file (it holds the teacher's scores on all training cases) ----
TEACHER_WAIT_MIN = 0
CROSSFIT_REPS = 6        # repetitions of the 50/50 split used for the honest threshold estimate

EPOCH_CHECKPOINTS = True   # True = after every epoch the full training state is saved to the bucket; Run All again after a stopped session resumes from the last finished epoch
MAX_SWEEP_HOURS = 1000        # do not START a setup that would not finish within this many hours (Kaggle stops a session at 12 h; finished setups are cached: Run All again resumes)
QUICK_TEST = False            # True = tiny end-to-end check (a handful of short documents, 1 epoch, 256-token windows, first 2 setups)
if QUICK_TEST:
    FULL_RUN = False; MAX_TRAIN_DOCS = 24; MAX_DEV_DOCS = 12
    SWEEP = [(n, dict(o, EPOCHS=1, EPOCH_DOCS=24, DEV_SUB=None, KD_TEACHER=None, KD_ALPHA=0.0)) for n, o in SWEEP[:2]]
    TEACHER_WAIT_MIN = 0
    BASE.update(WINDOW=256, STRIDE=256, MAX_CHUNKS=3, HEAD_CHUNKS=1, ACCUM=4)
    SWEEP_NAME += "_quicktest"
print(f"{SWEEP_NAME}: {len(SWEEP)} setups:", [n for n, _ in SWEEP])

p5_moe_lorma: 1 setups: ['moe_lorma_long']


## 2. Set-up: login, bucket

In [2]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.3.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


## 3. Phase-1 BPE tokenizer (class unchanged), loaded from storage

In [3]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


## 4. Phase-1 model code (copied unchanged from the Phase-2 notebook of this architecture)

In [4]:
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"   # as in the Phase-1 run
CONTEXT_LEN = 2048

class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# ---- Settings specific to this notebook ------------------------------
# ---- Mixture-of-Experts FFN (DeepSeekMoE style) -------------------------------------------------------------
MOE_N_ROUTED = 16          # routed experts per MoE layer (fine-grained: each is only 1/4 the width of the dense FFN)
MOE_N_SHARED = 1           # shared experts: ALWAYS applied to every token (they hold the common knowledge)
MOE_TOP_K = 3              # routed experts used per token. Active FFN width = (1 shared + 3 routed) x 768 = 3072 = the dense FFN's width,
                           # so the compute per token matches the dense GQA run; the TOTAL FFN capacity is 17 x 768 = 4.25x larger.
MOE_D_EXPERT = 768         # hidden width of one expert
MOE_DENSE_FIRST_LAYERS = 1 # the first layer stays a dense FFN (as in DeepSeek: early layers route poorly)
MOE_BIAS_RATE = 0.001      # aux-loss-free load balancing (DeepSeek-V3): per step, a per-expert selection bias moves by this much
                           # towards the under-used experts. The bias only affects WHICH experts are chosen, never the output weights.

GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class ExpertMLP(nn.Module):
    """One expert: the same Linear -> GELU -> Linear as the dense FFN, just narrower."""

    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up = nn.Linear(d_model, d_hidden)
        self.down = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class MoEFFN(nn.Module):
    """DeepSeekMoE feed-forward layer, written from scratch.

        y = sum_s Shared_s(x)  +  sum_{e in TopK(x)} g_e(x) * Routed_e(x)

    * router: affinity s_e = sigmoid(w_e . x) for every routed expert e (DeepSeek-V3 uses sigmoid, not softmax)
    * selection: the top-k experts by (s_e + bias_e); the bias is a NON-learned buffer used only for choosing
    * gates: the selected s_e, renormalised to sum to 1 (so the routed part has a fixed scale)
    * load balancing WITHOUT an auxiliary loss: in training, after each forward pass, bias_e moves by +-MOE_BIAS_RATE
      towards the under-loaded experts (overloaded: down, underloaded: up). Nothing is dropped (no capacity limit).
    Every token is processed independently, so the KV-cache decode path needs no change."""

    def __init__(self, d_model, n_routed, n_shared, top_k, d_expert, bias_rate):
        super().__init__()
        assert 1 <= top_k <= n_routed
        self.n_routed, self.top_k, self.bias_rate = n_routed, top_k, bias_rate
        self.router = nn.Linear(d_model, n_routed, bias=False)
        nn.init.normal_(self.router.weight, std=0.02)
        self.shared = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_shared)])
        self.experts = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_routed)])
        self.register_buffer("select_bias", torch.zeros(n_routed))      # saved in the checkpoint (it is part of the trained router)
        self.last_load = None                                           # tokens routed to each expert in the latest training step

    def forward(self, x):
        shape = x.shape
        flat = x.reshape(-1, shape[-1])                                  # (N, d): the router does not care about batch/position
        affinity = torch.sigmoid(self.router(flat).float())              # (N, E), fp32 even under autocast
        _, chosen = (affinity + self.select_bias).topk(self.top_k, dim=-1)       # (N, k): selection uses the bias ...
        gate = affinity.gather(1, chosen)                                # ... but the gate values do not
        gate = gate / gate.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(flat)
        for expert in self.shared:
            out = out + expert(flat)
        for e, expert in enumerate(self.experts):
            token_idx, slot = (chosen == e).nonzero(as_tuple=True)       # the tokens that picked expert e, and in which of their k slots
            if token_idx.numel() == 0:
                continue
            y = expert(flat[token_idx]) * gate[token_idx, slot].unsqueeze(-1).to(flat.dtype)
            out.index_add_(0, token_idx, y.to(out.dtype))
        if self.training:
            with torch.no_grad():
                load = torch.bincount(chosen.reshape(-1), minlength=self.n_routed).float()
                self.last_load = load
                self.select_bias += self.bias_rate * torch.sign(load.mean() - load)
        return out.reshape(shape)


def moe_load_stats(model):
    """[(layer index, busiest expert's load / the average (1.0 = perfectly balanced), number of experts with < 10% of an even share)]
    for every MoE layer, from the latest training step."""
    rows = []
    for i, blk in enumerate(model.blocks):
        if isinstance(blk.ff, MoEFFN) and blk.ff.last_load is not None:
            l = blk.ff.last_load.cpu()
            rows.append((i, round((l.max() / l.mean()).item(), 3), int((l < 0.1 * l.mean()).sum().item())))
    return rows


def moe_load_report(model):
    rows = moe_load_stats(model)
    if not rows:
        print("MoE load report: no training step has run in this session yet.")
        return
    print("MoE load balance (max expert load / mean, experts with <10% of an even share):")
    print("  " + "   ".join(f"layer {i}: {r:.2f}x / {d}" for i, r, d in rows))


def count_active_params(model):
    """Parameters used per token: everything except the routed experts that are NOT selected (top_k of n_routed are used)."""
    total = sum(p.numel() for p in model.parameters())
    inactive = 0
    for blk in model.blocks:
        if isinstance(blk.ff, MoEFFN):
            per_expert = sum(p.numel() for p in blk.ff.experts[0].parameters())
            inactive += per_expert * (blk.ff.n_routed - blk.ff.top_k)
    return total, total - inactive


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers, layer_idx=0):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        if layer_idx >= MOE_DENSE_FIRST_LAYERS:
            assert NORM_TYPE != "deepnorm", "the MoE variant is only set up for layernorm/rmsnorm"
            self.ff = MoEFFN(d_model, MOE_N_ROUTED, MOE_N_SHARED, MOE_TOP_K, MOE_D_EXPERT, MOE_BIAS_RATE)
        else:
            self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers, layer_idx=i) for i in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.fc_out(h)

## 5. Backbone wrapper (runs the Phase-1 blocks and returns hidden states)

In [5]:
# ===================== Backbone wrapper for the Phase-1 decoder models (GQA / MoE CachedDecoderModel) =====================
# The Phase-1 model code above is UNCHANGED. For NER we only (a) run its blocks and take the hidden states instead of the
# next-token logits, and (b) optionally replace the attention's forward with one that can drop the causal mask
# ("bidirectional") and/or prepend learned prefix keys/values (prefix tuning). With both options off it calls the
# original forward unchanged.

_ORIG_GQA_FORWARD = CachedGQAttention.forward


def ner_gqa_forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
    bidir = getattr(self, "ner_bidirectional", False)
    pk = getattr(self, "prefix_k", None)
    if not bidir and pk is None:
        return _ORIG_GQA_FORWARD(self, x, key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    if rope is not None:
        q = apply_rope(q, *rope)
        k = apply_rope(k, *rope)
    P = 0
    if pk is not None:                                          # learned prefix keys/values (no position: never rotated)
        P = pk.shape[1]
        k = torch.cat([pk.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
        v = torch.cat([self.prefix_v.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
    if self.group_size > 1:
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
    if bidir and P == 0 and key_padding_mask is None:          # full-attention fast path (no mask tensor at all)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
        return self.out_proj(out)
    pos = torch.arange(T, device=x.device)
    allowed = torch.ones(T, T, dtype=torch.bool, device=x.device) if bidir else (pos.unsqueeze(1) >= pos.unsqueeze(0))
    allowed = allowed.unsqueeze(0).expand(B, T, T)
    if key_padding_mask is not None:
        allowed = allowed & key_padding_mask.unsqueeze(1)
        allowed = allowed | torch.eye(T, dtype=torch.bool, device=x.device).unsqueeze(0)
    if P:
        allowed = torch.cat([torch.ones(B, T, P, dtype=torch.bool, device=x.device), allowed], dim=2)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed.unsqueeze(1))
    out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
    return self.out_proj(out)


CachedGQAttention.forward = ner_gqa_forward


class DecoderBackbone(nn.Module):
    def __init__(self, lm, bidirectional=False):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        self.causal = not bidirectional
        for blk in lm.blocks:
            blk.attn.ner_bidirectional = bidirectional
        if isinstance(lm.fc_out, nn.Sequential):                # the final LayerNorm is kept; the vocab projection is dropped
            self.final_norm = lm.fc_out[0]
        else:
            self.final_norm = nn.Identity()
        lm.fc_out = nn.Identity()
        self.d_model = lm.token_emb.embedding_dim
        self.max_len = lm.rope_cos.shape[0] if lm.use_rope else lm.pos_emb.num_embeddings

    def layer_list(self):
        return list(self.lm.blocks)

    def final_modules_params(self):
        return list(self.final_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        d = self.d_model
        if kind == "houlsby":                                   # after attention AND after the feed-forward, every block
            return [(m, d) for blk in self.lm.blocks for m in (blk.attn, blk.ff)]
        if kind == "pfeiffer":                                  # after the feed-forward only
            return [(blk.ff, d) for blk in self.lm.blocks]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        for blk in self.lm.blocks:
            a = blk.attn
            a.prefix_k = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)
            a.prefix_v = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)

    def features(self, ids, mask, soft_prompt=None):
        lm = self.lm
        h = lm.token_emb(ids)
        B = ids.shape[0]
        P = 0
        if soft_prompt is not None:
            P = soft_prompt.shape[0]
            h = torch.cat([soft_prompt.unsqueeze(0).expand(B, -1, -1).to(h.dtype), h], dim=1)
            mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=mask.device), mask], dim=1)
        T = h.shape[1]
        assert T <= self.max_len, f"window ({T} incl. {P} prompt tokens) longer than the model's {self.max_len} positions"
        pos = torch.arange(T, device=ids.device)
        rope = None
        if lm.use_rope:
            rope = (lm.rope_cos[pos].unsqueeze(0).unsqueeze(0), lm.rope_sin[pos].unsqueeze(0).unsqueeze(0))
        else:
            h = h + lm.pos_emb(pos).unsqueeze(0)
        # right padding: with causal attention real tokens never see the pads, so no mask is needed (fast fused kernel);
        # bidirectional attention needs the padding mask
        kpm = None if (self.causal or bool(mask.all())) else mask
        for blk in lm.blocks:
            h = blk(h, key_padding_mask=kpm, rope=rope)
        h = self.final_norm(h)
        return h[:, P:]


@torch.no_grad()
def lm_sanity_loss(lm, docs, device, n_windows=6, length=1024):
    """Next-token loss of the loaded Phase-1 model on a few NER-document windows (BEFORE the LM head is dropped).
    A correctly loaded model gives roughly its Phase-1 held-out loss (~2.5-3.5 nats); random weights give ~10.6."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:length + 1], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < 2:
            continue
        logits = lm(ids[:, :-1]).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="sum").item()
        cnt += ids.shape[1] - 1
    return tot / max(cnt, 1)


def build_phase1_lm():
    """The Phase-1 MoE exactly as trained (218.7M params), with its trained weights."""
    torch.manual_seed(0)
    lm = CachedDecoderModel(VOCAB, d_model=768, n_heads=12, n_layers=8, d_ff=3072, context_len=CONTEXT_LEN)
    lm.fc_out = nn.Sequential(make_norm(768, NORM_TYPE), lm.fc_out)          # the final LayerNorm (FINAL_NORM = True in Phase 1)
    lm.load_state_dict(PHASE1_STATE, strict=True)
    for m in lm.modules():
        if isinstance(m, MoEFFN):
            m.bias_rate = MOE_BIAS_RATE_FT
    return lm


def make_backbone(lm, C=None):
    return DecoderBackbone(lm, bidirectional=((C or {}).get("ATTENTION", ATTENTION) == "bidirectional"))


def phase1_sanity(lm, docs):
    return lm_sanity_loss(lm, docs, device)

## 6. Phase-5 engine: chunking, multi-label document head, losses, thresholds, training, inference, scoring

In [6]:
# ===================== Phase 5 (Legal Statute Identification) engine: long-document chunking, chunk pooling, multi-label document heads, losses, thresholds, training, inference =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
# A document = its BPE token ids. It is cut into CHUNKS of <= WINDOW tokens (sliding window, stride STRIDE); at most MAX_CHUNKS chunks are kept
# (SELECT says which). The backbone reads every chunk as one row, a chunk is pooled to one vector (mean of its tokens and/or its last token), a document
# head (mean / attention / BiLSTM + attention / one attention per statute) turns the chunk vectors into L statute scores (logits, one per statute id).
import math, time, random, json, os, copy, zipfile, hashlib
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents, chunks
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(r):
    t = r["text"]
    return " ".join(t) if isinstance(t, list) else str(t)


def text_hash(t):
    return hashlib.md5((t[:2000] + "||" + t[-2000:]).encode("utf-8", errors="ignore")).hexdigest()


def label_list(v):
    return sorted(set(int(x) for x in (v if isinstance(v, (list, tuple)) else [v])))


def multi_hot(labels, L):
    y = np.zeros(L, dtype=np.uint8)
    if len(labels):
        y[list(labels)] = 1
    return y


def encode_rows(tok, rows, id_key, label_key, max_tokens, log=print):
    """-> list of dict(id, ids=np.uint16 array, labs=list|None, n). Documents longer than max_tokens keep their first and last max_tokens//2 tokens."""
    out, t0, cut = [], time.time(), 0
    for i, r in enumerate(rows):
        ids = tok.encode(doc_text(r))
        if not ids:
            ids = [1]
        if len(ids) > max_tokens:
            ids = ids[:max_tokens // 2] + ids[-(max_tokens - max_tokens // 2):]; cut += 1
        out.append(dict(id=r[id_key], ids=np.asarray(ids, dtype=np.uint16), labs=label_list(r[label_key]) if label_key and r.get(label_key) is not None else None, n=len(ids)))
        if (i + 1) % 5000 == 0:
            log(f"    tokenised {i + 1}/{len(rows)} documents [{time.time() - t0:.0f}s]")
    return out, dict(documents=len(out), tokens=int(sum(d["n"] for d in out)), cut=cut)


def make_spans(n, C):
    """Chunk spans (start, end) of a document with n tokens, after applying MAX_CHUNKS / SELECT."""
    W, S = C["WINDOW"], max(1, C["STRIDE"])
    if n <= W:
        return [(0, n)]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n:
        starts.append(n - W)                                   # the last chunk is aligned to the end of the document
    spans = [(s, s + W) for s in starts]
    K, m, how = len(spans), C["MAX_CHUNKS"], C["SELECT"]
    if how == "all" or m is None or K <= m:
        return spans
    if how == "first":
        idx = list(range(m))
    elif how == "last":
        idx = list(range(K - m, K))
    elif how == "middle":
        a = (K - m) // 2; idx = list(range(a, a + m))
    elif how == "head_tail":
        h = min(C["HEAD_CHUNKS"], m - 1) if m > 1 else 0
        idx = list(range(h)) + list(range(K - (m - h), K))
    elif how == "uniform":
        idx = sorted(set(int(round(x)) for x in np.linspace(0, K - 1, m)))
    else:
        raise ValueError(how)
    return [spans[i] for i in idx]


def collate(docs, idxs, C, with_labels=True, pad_id=0):
    rows, counts, spans_all = [], [], []
    for di in idxs:
        d = docs[di]; sp = make_spans(d["n"], C)
        for s, e in sp:
            rows.append(d["ids"][s:e].tolist())
        counts.append(len(sp)); spans_all.append(sp)
    R, T = len(rows), max(len(r) for r in rows)
    ids = torch.full((R, T), pad_id, dtype=torch.long)
    mask = torch.zeros((R, T), dtype=torch.bool)
    for r, a in enumerate(rows):
        ids[r, :len(a)] = torch.tensor(a); mask[r, :len(a)] = True
    K = max(counts)
    chunk_mask = torch.arange(K).unsqueeze(0) < torch.tensor(counts).unsqueeze(1)
    b = dict(ids=ids, mask=mask, counts=counts, chunk_mask=chunk_mask, spans=spans_all)
    if with_labels and all(docs[di].get("y") is not None for di in idxs):
        b["labels"] = torch.from_numpy(np.stack([docs[di]["y"] for di in idxs])).float()
    if with_labels and all(docs[di].get("kd") is not None for di in idxs):
        b["kd"] = torch.stack([torch.as_tensor(docs[di]["kd"]).float() for di in idxs])
    return b


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 2. Model = backbone + chunk pooling + multi-label document head
# ----------------------------------------------------------------------------------------------------------------------
class DocHead(nn.Module):
    """chunk vectors (B, K, d_in) + chunk mask -> statute logits (B, L).
    kind: 'mean' | 'attn' (one attention pooling shared by all statutes) | 'bilstm' (BiLSTM over the chunk sequence, then shared attention pooling)
          | 'label_attn' (every statute has its own attention query over the chunks: it can look at the chunk that is about it).
    label_mix: a zero-initialised L x L layer on the sigmoid of the first-stage scores lets the head learn label co-occurrence (e.g. statute 12 -> 13)."""

    def __init__(self, d_in, n_labels, kind="bilstm", hidden=256, dropout=0.1, norm=True, label_mix=False):
        super().__init__()
        self.kind, self.L = kind, n_labels
        self.norm = nn.LayerNorm(d_in) if norm else nn.Identity()
        self.drop = nn.Dropout(dropout)
        d = d_in
        self.lstm = None
        if kind == "bilstm":
            self.lstm = nn.LSTM(d_in, hidden, batch_first=True, bidirectional=True)
            d = 2 * hidden
        if kind in ("attn", "bilstm"):
            self.score = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
            self.out = nn.Linear(d, n_labels)
        elif kind == "label_attn":
            self.key = nn.Sequential(nn.Linear(d, 128), nn.Tanh())
            self.lq = nn.Parameter(torch.randn(n_labels, 128) * 0.1)
            self.W = nn.Parameter(torch.randn(n_labels, d) * (1.0 / math.sqrt(d)))
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif kind == "mean":
            self.out = nn.Linear(d, n_labels)
        else:
            raise ValueError(kind)
        self.mix = nn.Linear(n_labels, n_labels, bias=False) if label_mix else None
        if self.mix is not None:
            nn.init.zeros_(self.mix.weight)
        self.last_attn = None

    @torch.no_grad()
    def init_bias(self, prior):
        """Output bias = logit of the label frequency: training starts from 'every statute at its base rate', not from p = 0.5 for all 100 statutes."""
        b = torch.logit(torch.as_tensor(prior, dtype=torch.float32).clamp(1e-4, 1 - 1e-4))
        (self.b if self.kind == "label_attn" else self.out.bias).copy_(b)

    def forward(self, v, mask):
        x = self.drop(self.norm(v.float()))
        if self.lstm is not None:
            lengths = mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            with torch.autocast(device_type=x.device.type, enabled=False):
                o, _ = self.lstm(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=v.shape[1])
        if self.kind == "label_attn":
            s = torch.einsum("bkh,lh->bkl", self.key(x), self.lq).masked_fill(~mask.unsqueeze(-1), float("-inf"))
            w = torch.softmax(s, dim=1)                                            # (B, K, L): each statute's attention over the chunks
            self.last_attn = w.mean(-1).detach()
            logits = (w * torch.einsum("bkd,ld->bkl", x, self.W)).sum(1) + self.b
        else:
            if self.kind == "mean":
                w = mask.float() / mask.float().sum(1, keepdim=True).clamp(min=1)
            else:
                s = self.score(x).squeeze(-1).masked_fill(~mask, float("-inf"))
                w = torch.softmax(s, dim=1)
            self.last_attn = w.detach()
            logits = self.out((x * w.unsqueeze(-1)).sum(1))
        if self.mix is not None:
            logits = logits + self.mix(torch.sigmoid(logits))
        return logits


class DocModel(nn.Module):
    """Attribute names shared with the Phase-3/4 model (backbone, soft_prompt, head_parameters) so the technique code works unchanged."""

    def __init__(self, backbone, d_model, C):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None
        self.pool = C["POOL"]
        self.row_group = C["ROW_GROUP"]
        self.ckpt = C.get("CKPT", False)
        if self.ckpt and any(getattr(m, "bias_rate", 0) > 0 or (hasattr(m, "tau") and getattr(type(m)._route, "__name__", "") != "_mor_route_frozen") for m in backbone.modules()):
            print("note: the backbone updates routing state inside its forward pass (MoE bias / MoR threshold): activation checkpointing switched off")
            self.ckpt = False
        d_pool = d_model * (2 if self.pool == "mean+last" else 1)
        self.head = DocHead(d_pool, C["N_LABELS"], C["DOC_HEAD"], C["HEAD_LSTM_HIDDEN"], C["HEAD_DROPOUT"], C["HEAD_NORM"], C.get("LABEL_MIX", False))

    def head_parameters(self):
        return list(self.head.parameters())

    def _pool_rows(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt).float()
        parts = []
        if self.pool in ("mean", "mean+last"):
            m = mask.unsqueeze(-1).float()
            parts.append((h * m).sum(1) / m.sum(1).clamp(min=1))
        if self.pool in ("last", "mean+last"):
            last = mask.sum(1) - 1
            parts.append(h[torch.arange(h.shape[0], device=h.device), last])
        return torch.cat(parts, -1)

    def chunk_vectors(self, b):
        ids, mask = b["ids"], b["mask"]
        trainable = self.training and any(p.requires_grad for p in self.backbone.parameters()) or (self.training and self.soft_prompt is not None)
        out = []
        for a in range(0, ids.shape[0], self.row_group):
            i, m = ids[a:a + self.row_group], mask[a:a + self.row_group]
            L = int(m.sum(1).max()); i, m = i[:, :L], m[:, :L]
            if not trainable:
                with torch.no_grad():
                    out.append(self._pool_rows(i, m))
            elif self.ckpt:
                out.append(checkpoint(self._pool_rows, i, m, use_reentrant=False))
            else:
                out.append(self._pool_rows(i, m))
        return torch.cat(out, 0)                                              # (R, d_pool), documents in order

    def forward(self, b):
        v = self.chunk_vectors(b)
        cm = b["chunk_mask"]
        vp = v.new_zeros(cm.shape[0], cm.shape[1], v.shape[1])
        vp[cm] = v                                                            # row-major order of True entries == document order
        return self.head(vp, cm)                                              # (B, L)


# ----------------------------------------------------------------------------------------------------------------------
# 3. Scores and thresholds (macro-F1 over the L statutes is the primary metric; micro-F1 and subset accuracy are secondary)
# ----------------------------------------------------------------------------------------------------------------------
def ml_scores(Y, pred):
    """Y, pred: (N, L) 0/1 arrays. Per-statute F1 is 0 when a statute has no true and no predicted positive (never the case here: every statute has dev support)."""
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tp = (Y & pred).sum(0).astype(np.float64); fp = (~Y & pred).sum(0); fn = (Y & ~pred).sum(0)
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0); r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    mp, mr = TP / max(TP + FP, 1), TP / max(TP + FN, 1)
    inter = (Y & pred).sum(1).astype(np.float64); den = Y.sum(1) + pred.sum(1)
    sample_f = np.divide(2 * inter, den, out=np.zeros_like(inter), where=den > 0)
    return dict(macro_F1=float(f.mean()), micro_F1=float(2 * mp * mr / max(mp + mr, 1e-12)), subset_acc=float((Y == pred).all(1).mean()), macro_P=float(p.mean()), macro_R=float(r.mean()),
                micro_P=float(mp), micro_R=float(mr), samples_F1=float(sample_f.mean()), hamming=float((Y != pred).mean()), mean_pred_labels=float(pred.sum(1).mean()),
                mean_true_labels=float(Y.sum(1).mean()), per_label_F1=f.tolist(), n=int(len(Y)))


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(np.asarray(x, dtype=np.float64), -30, 30)))


GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)


def apply_thr(P, thr, min_one=True):
    """P (N, L) probabilities, thr scalar or (L,). Every case has at least one statute in the data: if none passes its threshold, the statute with the largest margin P - thr is predicted."""
    thr = np.broadcast_to(np.asarray(thr, dtype=np.float64), (P.shape[1],))
    pred = P > thr
    if min_one:
        none = ~pred.any(1)
        if none.any():
            am = (P[none] - thr).argmax(1); sub = np.zeros((none.sum(), P.shape[1]), dtype=bool); sub[np.arange(len(am)), am] = True; pred[none] = sub
    return pred


def f1_grid(Y, P, grid=GRID):
    """-> F (T, L): per-statute F1 at every global threshold of the grid."""
    Yb = np.asarray(Y).astype(bool); F_ = np.zeros((len(grid), Yb.shape[1]))
    ny = Yb.sum(0).astype(np.float64)
    for k, t in enumerate(grid):
        pr = P > t
        tp = (Yb & pr).sum(0); npred = pr.sum(0)
        F_[k] = np.divide(2.0 * tp, ny + npred, out=np.zeros(Yb.shape[1]), where=(ny + npred) > 0)
    return F_


def fit_thresholds(Y, P, mode="global", shrink=0.5, grid=GRID, smooth=5):
    """-> (L,) thresholds. global: one threshold for all statutes (the one maximising macro-F1); per_label: each statute its own best threshold (on a
    smoothed F1 curve, to avoid chasing noise); shrunk: per_label pulled towards the global threshold (shrink = weight of the per-label value)."""
    F_ = f1_grid(Y, P, grid); L = F_.shape[1]
    tg = float(grid[int(np.argmax(F_.mean(1)))])
    if mode == "global":
        return np.full(L, tg)
    k = max(1, smooth)
    Fs = np.apply_along_axis(lambda c: np.convolve(np.pad(c, (k // 2, k // 2), mode="edge"), np.ones(k) / k, mode="valid"), 0, F_) if k > 1 else F_
    tl = grid[np.argmax(Fs, 0)]
    return tl if mode == "per_label" else shrink * tl + (1 - shrink) * tg


THR_MODES = ("global", "per_label", "shrunk")


def cross_fit(Y, P, modes=THR_MODES, reps=6, min_one=True, seed=0):
    """Honest threshold evaluation: thresholds are fitted on one random half of the cases and scored on the other half. -> {mode: mean macro-F1}, {mode: sd}."""
    res = {m: [] for m in modes}
    for r in range(reps):
        perm = np.random.RandomState(seed + r).permutation(len(Y)); A, B = perm[:len(Y) // 2], perm[len(Y) // 2:]
        for m in modes:
            for tr, te in ((A, B), (B, A)):
                res[m].append(ml_scores(Y[te], apply_thr(P[te], fit_thresholds(Y[tr], P[tr], m), min_one))["macro_F1"])
    return {m: float(np.mean(v)) for m, v in res.items()}, {m: float(np.std(v)) for m, v in res.items()}


def best_global(Y, P, min_one=True):
    t = float(fit_thresholds(Y, P, "global")[0])
    return t, ml_scores(Y, apply_thr(P, t, min_one))["macro_F1"]


# ----------------------------------------------------------------------------------------------------------------------
# 4. Losses
# ----------------------------------------------------------------------------------------------------------------------
def pos_weights(Ytrain, power, cap):
    """pos_weight_l = ((N - n_l) / n_l) ** power, capped (power 0.5 = inverse square root of the odds)."""
    n = Ytrain.sum(0).astype(np.float64); N = len(Ytrain)
    w = np.minimum(((N - n) / np.maximum(n, 1)) ** power, cap)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def ml_loss(logits, y, C, pw=None):
    """LOSS: bce | bce_pw (pos_weight per statute) | focal | asl (asymmetric loss, Ridnik et al. 2021). Returns the mean over the batch of the per-statute mean x LOSS_SCALE."""
    eps = C.get("LABEL_SMOOTHING", 0.0)
    if eps:
        y = y * (1 - eps) + eps / 2
    kind = C["LOSS"]
    if kind in ("bce", "bce_pw"):
        l = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pw if kind == "bce_pw" else None, reduction="none")
    elif kind == "focal":
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none"); p = torch.sigmoid(logits)
        pt = y * p + (1 - y) * (1 - p)
        l = bce * (1 - pt) ** C["FOCAL_GAMMA"]
    elif kind == "asl":
        xp = torch.sigmoid(logits); xn = 1 - xp
        if C["ASL_CLIP"] > 0:
            xn = (xn + C["ASL_CLIP"]).clamp(max=1)
        lp = y * torch.log(xp.clamp(min=1e-8)); ln = (1 - y) * torch.log(xn.clamp(min=1e-8))
        pt = xp * y + xn * (1 - y)
        g = C["ASL_GAMMA_POS"] * y + C["ASL_GAMMA_NEG"] * (1 - y)
        l = -(lp + ln) * (1 - pt).detach().pow(g)
    else:
        raise ValueError(kind)
    return l.mean(1).mean() * C["LOSS_SCALE"]


def kd_loss(logits, teacher_logits, T, C):
    """Per-statute distillation: the student's tempered sigmoid is pulled to the teacher's tempered sigmoid (binary cross-entropy with soft targets, x T^2)."""
    soft = torch.sigmoid(teacher_logits / T)
    return F.binary_cross_entropy_with_logits(logits / T, soft, reduction="none").mean(1).mean() * T * T * C["LOSS_SCALE"]


# ----------------------------------------------------------------------------------------------------------------------
# 5. Training
# ----------------------------------------------------------------------------------------------------------------------
def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


@torch.no_grad()
def predict_logits(model, docs, C, device, collect_attn=False):
    """-> (N, L) float32 logits in the order of docs, plus timing; with collect_attn also the per-document chunk attention weights and spans."""
    model.eval(); model.to(device)
    order = sorted(range(len(docs)), key=lambda i: len(make_spans(docs[i]["n"], C)))        # similar chunk counts together: less padding
    out = np.zeros((len(docs), C["N_LABELS"]), dtype=np.float32)
    attn = {} if collect_attn else None
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    t0, tokens = time.time(), 0
    for a in range(0, len(order), C["INFER_BATCH"]):
        idx = order[a:a + C["INFER_BATCH"]]
        b = to_device(collate(docs, idx, C, with_labels=False), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(b)
        out[idx] = lg.float().cpu().numpy(); tokens += int(b["mask"].sum())
        if collect_attn:
            w = model.head.last_attn.cpu().numpy()
            for j, di in enumerate(idx):
                attn[docs[di]["id"]] = dict(w=w[j, :b["counts"][j]].tolist(), spans=b["spans"][j], n=docs[di]["n"])
    if device.type == "cuda":
        torch.cuda.synchronize()
    secs = time.time() - t0
    info = dict(seconds=round(secs, 2), tokens=tokens, tokens_per_sec=round(tokens / max(secs, 1e-9), 1), docs_per_sec=round(len(docs) / max(secs, 1e-9), 2))
    return (out, info, attn) if collect_attn else (out, info)


def train_doc(model, train_docs, dev_docs, C, device, log=print, dev_sub=None, ckpt_save=None, ckpt_load=None):
    """Trains on train_docs in EPOCHS 'epochs' of EPOCH_DOCS documents each (a stream over a reshuffled permutation, so a huge training set can be seen
    partially). After every epoch a fixed sample of the selection set is scored; the best epoch's state is kept (SELECT_METRIC: f1_tuned = macro-F1 at the
    best global threshold, f1 = at 0.5). Optional model hooks (AdaLoRA): extra_loss(), on_before_step(step, total)."""
    model.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    N = len(train_docs)
    per_epoch = min(C["EPOCH_DOCS"] or N, N)
    epochs = C["EPOCHS"]
    Ytr = np.stack([d["y"] for d in train_docs])
    pw = pos_weights(Ytr, C["PW_POWER"], C["PW_CAP"]).to(device) if C["LOSS"] == "bce_pw" else None
    model.head.init_bias(Ytr.mean(0))
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(per_epoch / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    kd_on = C.get("KD_ALPHA", 0) > 0 and all(d.get("kd") is not None for d in train_docs)
    rng = random.Random(C["SEED"] * 1000 + 7)
    stream = []
    def take(k):
        nonlocal stream
        while len(stream) < k:
            p = list(range(N)); rng.shuffle(p); stream += p
        out, stream = stream[:k], stream[k:]
        return out
    dev_idx = list(range(len(dev_docs)))
    if dev_sub and dev_sub < len(dev_docs):
        dev_idx = sorted(random.Random(1).sample(dev_idx, dev_sub))
    dev_eval = [dev_docs[i] for i in dev_idx]
    Yd = np.stack([d["y"] for d in dev_eval])
    hist, best, best_state, best_ep, bad = [], -1.0, None, 0, 0
    t0, tokens, gstep = time.time(), 0, 0
    start_ep = 1
    ck = ckpt_load() if ckpt_load else None
    if ck is not None:          # resume after the last finished epoch (a previous session was stopped)
        load_trainable_state(model, ck["model"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        rng.setstate(ck["rng"]); stream = ck["stream"]; torch.set_rng_state(ck["torch_rng"])
        if device.type == "cuda" and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state(ck["cuda_rng"], device)
        hist, best, best_state, best_ep, bad = ck["hist"], ck["best"], ck["best_state"], ck["best_ep"], ck["bad"]
        tokens, gstep = ck["tokens"], ck["gstep"]; t0 = time.time() - ck["elapsed"]; start_ep = ck["epoch"] + 1
        log(f"  resumed from the checkpoint after epoch {ck['epoch']} (best so far: epoch {best_ep}, {best:.4f})")
    log(f"training: {N} train documents, {per_epoch} per epoch x {epochs} epochs, {steps_per_epoch} optimizer steps/epoch, loss {C['LOSS']}, head {C['DOC_HEAD']}" + (", KD on" if kd_on else "")
        + (f"; pos_weight range {float(pw.min()):.2f}-{float(pw.max()):.2f}" if pw is not None else ""))
    for ep in range(start_ep, epochs + 1):
        model.train()
        order = take(per_epoch)
        run_loss, nb, skipped, pending = 0.0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            b = to_device(collate(train_docs, order[bi:bi + C["BATCH"]], C), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float()
            loss = ml_loss(logits, b["labels"], C, pw)
            if kd_on:
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_loss(logits, b["kd"], C["KD_TEMPERATURE"], C)
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; tokens += int(b["mask"].sum()); run_loss += loss.item()
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"):
                    model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), train_seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        lg, einfo = predict_logits(model, dev_eval, C, device)
        P = sigmoid_np(lg)
        s05 = ml_scores(Yd, apply_thr(P, 0.5)); thr, ft = best_global(Yd, P)
        sc_tuned = ml_scores(Yd, apply_thr(P, thr))
        score = s05["macro_F1"] if C["SELECT_METRIC"] == "f1" else ft
        row.update(dev_macroF1=round(s05["macro_F1"], 4), dev_macroF1_tuned=round(ft, 4), tuned_thr=round(thr, 2), dev_microF1_tuned=round(sc_tuned["micro_F1"], 4),
                   dev_subset_acc_tuned=round(sc_tuned["subset_acc"], 4), eval_seconds=einfo["seconds"])
        if score > best:
            best, best_ep, best_state, bad = score, ep, trainable_state(model), 0
        else:
            bad += 1
        hist.append(row)
        log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if ckpt_save and ep < epochs:
            ckpt_save(dict(epoch=ep, model=trainable_state(model), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                           rng=rng.getstate(), stream=list(stream), torch_rng=torch.get_rng_state(),
                           cuda_rng=torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
                           hist=hist, best=best, best_state=best_state, best_ep=best_ep, bad=bad, tokens=tokens, gstep=gstep, elapsed=time.time() - t0))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping"); break
        if C["PATIENCE"] and bad >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad} epochs"); break
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget of {C['TIME_BUDGET_MIN']} min reached: stopping after epoch {ep}"); break
    train_secs = time.time() - t0 - sum(r["eval_seconds"] for r in hist)
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(train_secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 6. Submission
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ids, pred_matrix, out_dir, name, expected_ids, n_labels):
    """JSON list [{"id": ..., "pred": [statute ids]}] (one entry per test case, sorted unique integers in 0..L-1) + a zip with prediction.json at its root."""
    assert list(ids) == list(expected_ids) or set(ids) == set(expected_ids), "ids differ from test.json"
    assert len(set(ids)) == len(ids) == len(expected_ids), "duplicate or missing ids"
    pm = np.asarray(pred_matrix).astype(bool); assert pm.shape == (len(ids), n_labels)
    sub = [dict(id=i, pred=[int(x) for x in np.flatnonzero(row)]) for i, row in zip(ids, pm)]
    assert all(len(set(s["pred"])) == len(s["pred"]) and all(isinstance(x, int) and 0 <= x < n_labels for x in s["pred"]) for s in sub)
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json")
    with open(jpath, "w", encoding="utf-8") as f:
        json.dump(sub, f)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath


# ===================== The Phase-5 evaluator (from the course file evaluate_phase5.py; read_truth / read_pred / the three scores are unchanged, main() became a function) =====================
# Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]      Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]
import argparse, json, sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def official_eval(truth_path, prediction_path, verbose=True):
    """Same checks and the same three numbers as evaluate_phase5.py: Macro-F1, Micro-F1, Subset Accuracy."""
    t,n=read_truth(truth_path); p=read_pred(prediction_path,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    res=dict(examples=len(t), labels=n, macro_F1=float(f1_score(yt,yp,average="macro",zero_division=0)),
             micro_F1=float(f1_score(yt,yp,average='micro',zero_division=0)), subset_acc=float(accuracy_score(yt,yp)))
    if verbose:
        print(f'  [official evaluator] Examples: {res["examples"]}  Labels: {res["labels"]}  Macro-F1: {res["macro_F1"]:.6f}  Micro-F1: {res["micro_F1"]:.6f}  Subset Accuracy: {res["subset_acc"]:.6f}')
    return res


def check_test_file(pred_path, test_ids, n_labels):
    """The checks Codabench applies to a submission (read_pred) + exactly one object per test case."""
    p = read_pred(pred_path, n_labels)
    if set(p) != set(test_ids) or len(p) != len(test_ids):
        raise ValueError(f"ID mismatch with test.json: missing={len(set(test_ids) - set(p))}, extra={len(set(p) - set(test_ids))}")
    return True


def dev_codabench_check(ids, Y, pred, out_dir, name, verbose=True):
    """Writes the dev truth (as {"id", "gold"}) and the predictions (as {"id", "pred"}) as files and scores them with the official evaluator code.
    Run `python evaluate_phase5.py --truth <truth> --prediction <prediction>` on the same files to get the same numbers."""
    os.makedirs(out_dir, exist_ok=True)
    assert all(isinstance(i, str) for i in ids), "case ids must be strings (the evaluator requires it)"
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tpath, ppath = os.path.join(out_dir, "dev_truth.json"), os.path.join(out_dir, f"{name}_dev_prediction.json")
    json.dump([dict(id=i, gold=[int(x) for x in np.flatnonzero(r)]) for i, r in zip(ids, Y)], open(tpath, "w"))
    json.dump([dict(id=i, pred=[int(x) for x in np.flatnonzero(r)]) for i, r in zip(ids, pred)], open(ppath, "w"))
    return official_eval(tpath, ppath, verbose), tpath, ppath

## 7. Adaptation techniques (LoRA, LoRMA, AdaLoRA, IA3, QLoRA ...)

In [7]:
# ===================== Phase 3 adaptation techniques: frozen, full FT, LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix =====================
# All of them leave the Phase-1 model code untouched: they wrap/replace nn.Linear modules inside the backbone, or add small trainable tensors.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------------------------------------------------
# LoRA / rsLoRA / LoRMA
# ----------------------------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2022): y = W0 x + b + s * B A x,  A (r, in) random, B (out, r) zeros -> starts exactly at W0.
    s = alpha / r (LoRA)  or  alpha / sqrt(r) (rank-stabilised LoRA, Kalajdzievski 2023: keeps the update size stable when r grows)."""

    def __init__(self, base, r, alpha, dropout=0.0, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_ROT_IDX = {}


def _row_rotation_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                      # Mpi[i, j] = M[i, (j - i) mod d]  (row i rotated right by i)
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel, Modi, ACL Findings 2025): a MULTIPLICATIVE low-rank update of a frozen W0 (out x in).
      "plus" (LoRMA+):  h = ((alpha/r) B A + I) W0 x + b,   A (r, out) random, B (out, r) zeros -> starts at W0. Computed as
                        y0 = W0 x;  y0 + (alpha/r) B (A y0), so the out x out matrix is never formed.
      "pi"   (LoRMA_pi): h = I_pi(B A) W0 x + b, I_pi rotates row i of BA cyclically by i (rank inflation). Init so that I_pi(BA) = I."""

    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        self.base, self.variant, self.r, self.scale = base, variant, r, alpha / r
        d = base.out_features
        if variant == "plus":
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        elif variant == "pi":
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.randn(d, r) * (1.0 / math.sqrt(d)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
        else:
            raise ValueError(variant)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        else:
            M = self.B @ self.A
            Mpi = torch.gather(M, 1, _row_rotation_index(M.shape[0], M.device))
            y = F.linear(y0, Mpi.to(y0.dtype))
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


# ----------------------------------------------------------------------------------------------------------------------
# AdaLoRA (Zhang et al. 2023), simplified
# ----------------------------------------------------------------------------------------------------------------------
class AdaLoRALinear(nn.Module):
    """y = W0 x + b + s * P diag(lam * mask) Q x   (an SVD-shaped update: P (out, r) and Q (r, in) start orthonormal, lam (r) starts at 0).
    The controller below prunes whole (P_i, lam_i, Q_i) triplets by importance during training, moving the rank budget to the layers that
    need it. Simplification: a pruned triplet stays pruned."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        out, inn = base.out_features, base.in_features
        P = torch.linalg.qr(torch.randn(out, r))[0]                      # orthonormal columns
        Q = torch.linalg.qr(torch.randn(inn, r))[0].t().contiguous()     # orthonormal rows
        self.P, self.Q = nn.Parameter(P), nn.Parameter(Q)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("ada_mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.Q) * (self.lam * self.ada_mask)
        return self.base(x) + self.scale * F.linear(z, self.P)

    def orth_penalty(self):
        eye = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.t() - eye) ** 2).sum()


class AdaLoRAController:
    """Importance score of a triplet i: s(lam_i) + mean_j s(P_ji) + mean_j s(Q_ij), with s(w) = smoothed |w * grad(w)| times its
    smoothed uncertainty (paper, eq. 8-10). Budget schedule: full budget for the first t_i steps, then a cubic decrease to the target
    budget, then fixed for the last t_f steps. Also adds the orthogonality penalty to the loss."""

    def __init__(self, modules, target_rank, ti_frac=0.1, tf_frac=0.2, delta_t=10, beta=0.85, orth_weight=0.1):
        self.mods = modules
        self.r0 = modules[0].r
        self.b0 = len(modules) * self.r0
        self.bT = int(round(len(modules) * target_rank))
        self.ti_frac, self.tf_frac, self.dt, self.beta, self.orth = ti_frac, tf_frac, delta_t, beta, orth_weight
        z = lambda p: torch.zeros_like(p.data)
        self.st = [{n: (z(getattr(m, n)), z(getattr(m, n))) for n in ("lam", "P", "Q")} for m in modules]

    def extra_loss(self):
        return self.orth * sum(m.orth_penalty() for m in self.mods) / len(self.mods)

    def budget(self, step, total):
        ti, tf = int(self.ti_frac * total), int(self.tf_frac * total)
        if step < ti:
            return self.b0
        if step > total - tf:
            return self.bT
        frac = (step - ti) / max(1, total - ti - tf)
        return int(round(self.bT + (self.b0 - self.bT) * (1 - frac) ** 3))

    @torch.no_grad()
    def on_before_step(self, step, total):
        b = self.beta
        for m, st in zip(self.mods, self.st):
            for n, (ibar, u) in list(st.items()):
                p = getattr(m, n)
                if ibar.device != p.device:                       # the statistics were created on the CPU, the model may have moved since
                    ibar, u = ibar.to(p.device), u.to(p.device); st[n] = (ibar, u)
                if p.grad is None:
                    continue
                imp = (p.data * p.grad).abs()
                ibar.mul_(b).add_(imp, alpha=1 - b)
                u.mul_((b)).add_((imp - ibar).abs(), alpha=1 - b)
        tf = int(self.tf_frac * total)
        if step % self.dt == 0 and step <= total - tf:
            scores = []
            for m, st in zip(self.mods, self.st):
                s = lambda n: st[n][0] * st[n][1]
                scores.append(s("lam") + s("P").mean(0) + s("Q").mean(1))
            allsc = torch.cat(scores)
            keep = min(max(self.budget(step, total), 1), allsc.numel())
            thr = torch.sort(allsc, descending=True).values[keep - 1]
            for m, sc in zip(self.mods, scores):
                m.ada_mask.copy_((sc >= thr).float() * m.ada_mask)       # a pruned triplet stays pruned

    def rank_report(self):
        return [int(m.ada_mask.sum().item()) for m in self.mods]


# ----------------------------------------------------------------------------------------------------------------------
# QLoRA (Dettmers et al. 2023): frozen 4-bit NormalFloat base weights (+ double quantisation of the block constants) with LoRA on top
# ----------------------------------------------------------------------------------------------------------------------
NF4_TABLE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
                          -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                          0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """A frozen Linear stored as 4-bit NF4 codes (two per byte) with one absmax constant per `block` weights; the constants are themselves
    stored in 8 bit (double quantisation). The weight is rebuilt on the fly in every forward pass (pure PyTorch: no bitsandbytes)."""

    def __init__(self, base, block=64, double_quant=True, dq_block=256):
        super().__init__()
        self.in_features, self.out_features, self.block, self.double_quant = base.in_features, base.out_features, block, double_quant
        W = base.weight.detach().float().reshape(-1)
        self.numel = W.numel()
        Wp = torch.cat([W, W.new_zeros((-self.numel) % block)])
        blocks = Wp.view(-1, block)
        absmax = blocks.abs().amax(1).clamp(min=1e-8)
        flat = (blocks / absmax[:, None]).reshape(-1)
        codes = torch.empty(flat.numel(), dtype=torch.uint8)
        for i in range(0, flat.numel(), 1 << 21):
            ch = flat[i:i + (1 << 21)]
            codes[i:i + ch.numel()] = (ch.unsqueeze(1) - NF4_TABLE).abs().argmin(1).to(torch.uint8)
        self.register_buffer("packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("nf4", NF4_TABLE.clone(), persistent=False)
        self.n_blocks = absmax.numel()
        if double_quant:
            mean = absmax.mean()
            am = absmax - mean
            amp = torch.cat([am, am.new_zeros((-am.numel()) % dq_block)]).view(-1, dq_block)
            sc = amp.abs().amax(1).clamp(min=1e-12)
            self.register_buffer("am_q", torch.round(amp / sc[:, None] * 127).to(torch.int8))
            self.register_buffer("am_scale", sc)
            self.register_buffer("am_mean", mean.reshape(1))
        else:
            self.register_buffer("absmax", absmax)
        self.register_buffer("bias", base.bias.detach().clone() if base.bias is not None else None)

    def _absmax(self):
        if self.double_quant:
            return (self.am_q.float() / 127 * self.am_scale[:, None]).reshape(-1)[:self.n_blocks] + self.am_mean
        return self.absmax

    @torch.no_grad()
    def dequant(self, dtype=torch.float32):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0x0F).long()], 1).reshape(-1)
        vals = self.nf4[codes].view(-1, self.block) * self._absmax()[:, None]
        return vals.reshape(-1)[:self.numel].view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        W = self.dequant(x.dtype)
        return F.linear(x, W, None if self.bias is None else self.bias.to(x.dtype))


# ----------------------------------------------------------------------------------------------------------------------
# IA3 (Liu et al. 2022): learned per-channel rescaling vectors, initialised to 1
# ----------------------------------------------------------------------------------------------------------------------
class IA3Out(nn.Module):
    """y = l * Linear(x): rescales the OUTPUT channels (used on the key and value projections)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """y = Linear(l * x): rescales the INPUT channels (used on the feed-forward down-projection, i.e. the hidden activation)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


# ----------------------------------------------------------------------------------------------------------------------
# Applying a technique
# ----------------------------------------------------------------------------------------------------------------------
def _swap_linears(root, fn):
    """fn(qualified_name, nn.Linear) -> replacement module or None. Returns the names that were replaced."""
    done = []
    for qual, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear):
                full = f"{qual}.{cname}" if qual else cname
                new = fn(full, child)
                if new is not None:
                    setattr(mod, cname, new); done.append(full)
    return done


def _hit(full, suffixes):
    return any(full.endswith(s) for s in suffixes)


def size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6 + sum(b.numel() * b.element_size() for b in model.buffers()) / 1e6


def apply_technique(model, C, log=print):
    """Freezes/unfreezes the backbone and inserts modules according to C['TECHNIQUE']; the sentence head is always trainable."""
    tech, bb = C["TECHNIQUE"], model.backbone
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    T = C.get("PEFT_TARGETS", [])
    if tech == "frozen":
        pass                                                           # Phase-1 weights untouched: only the head learns (control)
    elif tech == "full_ft":
        for p in bb.parameters():
            p.requires_grad = True
    elif tech in ("lora", "rslora"):
        done = _swap_linears(bb, lambda n, l: LoRALinear(l, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0), tech == "rslora") if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], scale=(C["ALPHA"] / math.sqrt(C["RANK"]) if tech == "rslora" else C["ALPHA"] / C["RANK"]))
    elif tech == "lorma":
        done = _swap_linears(bb, lambda n, l: LoRMALinear(l, C["RANK"], C["ALPHA"], C["LORMA_VARIANT"]) if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], variant=C["LORMA_VARIANT"])
    elif tech == "adalora":
        mods = []

        def mk(n, l):
            if _hit(n, T):
                m = AdaLoRALinear(l, C["ADALORA_INIT_RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0)); mods.append(m); return m
        done = _swap_linears(bb, mk)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        ctrl = AdaLoRAController(mods, C["ADALORA_TARGET_RANK"], delta_t=C.get("ADALORA_DELTA_T", 10), orth_weight=C.get("ADALORA_ORTH", 0.1))
        model.extra_loss, model.on_before_step, model.ada_controller = ctrl.extra_loss, ctrl.on_before_step, ctrl
        info.update(targets=len(done), init_rank=C["ADALORA_INIT_RANK"], target_rank=C["ADALORA_TARGET_RANK"])
    elif tech == "qlora":
        blocks = bb.layer_list()
        n_q, n_l = [0], [0]

        def mk(n, l):
            q = NF4Linear(l, C.get("NF4_BLOCK", 64), C.get("NF4_DOUBLE_QUANT", True))
            n_q[0] += 1
            if _hit(n, T):
                n_l[0] += 1
                return LoRALinear(q, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0))
            return q
        for blk in blocks:
            _swap_linears(blk, mk)
        assert n_l[0], f"no linear layer matches PEFT_TARGETS={T}"
        info.update(quantized_linears=n_q[0], lora_targets=n_l[0], rank=C["RANK"], alpha=C["ALPHA"])
    elif tech == "ia3":
        n_out, n_in = [0], [0]

        def mk(n, l):
            if _hit(n, C["IA3_OUT"]):
                n_out[0] += 1; return IA3Out(l)
            if _hit(n, C["IA3_IN"]):
                n_in[0] += 1; return IA3In(l)
        _swap_linears(bb, mk)
        assert n_out[0] + n_in[0], f"no linear layer matches IA3_OUT={C['IA3_OUT']} / IA3_IN={C['IA3_IN']}"
        info.update(out_scaled=n_out[0], in_scaled=n_in[0])
    elif tech == "prompt":
        P = C["PROMPT_LEN"]
        emb = bb.embedding_weight()
        idx = torch.randint(100, emb.shape[0], (P,), generator=torch.Generator().manual_seed(0))
        model.soft_prompt = nn.Parameter(emb[idx].detach().clone().float())
        info.update(prompt_len=P)
    elif tech == "prefix":
        bb.enable_prefix(C["PREFIX_LEN"], 0)
        info.update(prefix_len=C["PREFIX_LEN"])
    else:
        raise ValueError(f"unknown TECHNIQUE {tech}")
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); "
        f"stored size {info['model_size_mb']} MB")
    return info

## 8. BitFit, partial fine-tuning, adapters

In [8]:
# ===================== Phase 4 additions to the adaptation techniques: BitFit, partial (top-layer) fine-tuning, bottleneck adapters =====================
# Wraps the Phase-3 apply_technique (LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix, full_ft, frozen are unchanged).
_p3_apply_technique = apply_technique


class BottleneckAdapter(nn.Module):
    """Houlsby/Pfeiffer-style adapter: y + W_up act(W_down y); W_up starts at zero so the model starts exactly at the Phase-1 function."""

    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, y):
        return y + self.up(F.gelu(self.down(y)))


def _attach_adapter(m, d, r):
    ad = BottleneckAdapter(d, r)
    m.add_module("adapter", ad)
    orig = m.forward
    def fwd(*a, **k):
        return m.adapter(orig(*a, **k))
    m.forward = fwd
    return ad


def apply_technique(model, C, log=print):
    tech, bb = C["TECHNIQUE"], model.backbone
    if tech not in ("bitfit", "partial", "adapters"):
        return _p3_apply_technique(model, C, log)
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    if tech == "bitfit":                                   # Ben Zaken et al. 2022: only the bias terms of the backbone are trained
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith("bias") or ".bias_" in name or name.endswith(("bias_ih_l0", "bias_hh_l0")):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "this backbone has no bias terms"
        info.update(bias_params=n)
    elif tech == "partial":                                # layer-wise fine-tuning: the top PARTIAL_LAYERS layers + the final norm
        layers = bb.layer_list(); k = C["PARTIAL_LAYERS"]
        for l in layers[-k:]:
            for p in l.parameters():
                p.requires_grad = True
        for p in bb.final_modules_params():
            p.requires_grad = True
        info.update(unfrozen_layers=k, of_layers=len(layers))
    else:                                                  # adapters
        sites = bb.adapter_sites(C["ADAPTER_KIND"])
        for m, d in sites:
            _attach_adapter(m, d, C["ADAPTER_R"])
        info.update(adapter_kind=C["ADAPTER_KIND"], adapter_sites=len(sites), bottleneck=C["ADAPTER_R"])
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); stored size {info['model_size_mb']} MB")
    return info

## 9. Data: Phase_5_LSI, tokenisation (cached in the bucket)

In [9]:
import numpy as np
P5 = "Phase_5_LSI"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)
RAW = {sp: load_json(get_dataset_file(f"{P5}/{d}/{sp}.json")) for sp, d in (("train", "Train"), ("dev", "Train"), ("test", "Test"))}
assert VOCAB < 65536, "token ids are stored as uint16"
ID_KEY, LABEL_KEY = "id", "labels"
assert LABEL_KEY in RAW["train"][0] and LABEL_KEY in RAW["dev"][0] and LABEL_KEY not in RAW["test"][0], "unexpected fields: " + str(list(RAW["train"][0].keys()))
print({sp: len(v) for sp, v in RAW.items()}, "| id:", ID_KEY, "| labels:", LABEL_KEY)
for sp in RAW:
    assert len({r[ID_KEY] for r in RAW[sp]}) == len(RAW[sp]), f"duplicate ids in {sp}"
assert not (set(r[ID_KEY] for r in RAW["train"]) & set(r[ID_KEY] for r in RAW["test"])), "a test id is also a training id"
TEST_IDS = [r[ID_KEY] for r in RAW["test"]]
_all = sorted({l for sp in ("train", "dev") for r in RAW[sp] for l in label_list(r[LABEL_KEY])})
N_LABELS = max(_all) + 1
assert _all == list(range(N_LABELS)), "statute ids are not contiguous 0..L-1"
BASE["N_LABELS"] = N_LABELS
print(f"{N_LABELS} statutes (ids 0..{N_LABELS - 1})")

# Cases of dev whose text also occurs in test (67 in the released files): never used to choose an epoch, a threshold or an ensemble. (Training cases that are also test cases are kept: they are legitimate training data.)
_test_h = {text_hash(doc_text(r)) for r in RAW["test"]}
DEV_IS_TEST_TEXT = {r[ID_KEY]: (text_hash(doc_text(r)) in _test_h) for r in RAW["dev"]}
print(f"NOTE: {sum(DEV_IS_TEST_TEXT.values())} dev cases have the same text as a test case: they are excluded from every selection step (selection set = the other {len(DEV_IS_TEST_TEXT) - sum(DEV_IS_TEST_TEXT.values())} dev cases).")
del _test_h

def subset(rows, n, seed=0):
    """A fixed shuffle; the first n. Prefix-consistent: the subset of size n is contained in the subset of any larger size."""
    if n is None or n >= len(rows):
        return rows
    perm = random.Random(seed).sample(range(len(rows)), len(rows))
    return [rows[i] for i in sorted(perm[:n])]

N_TRAIN = None if FULL_RUN else MAX_TRAIN_DOCS
WANT = dict(train=subset(RAW["train"], N_TRAIN), dev=subset(RAW["dev"], MAX_DEV_DOCS), test=RAW["test"])
N_ALL_TRAIN = len(RAW["train"])


def tokenised(split):
    """Tokenised cases of a split. The full-split token cache (p5tok_<split>_<MAX_DOC_TOKENS>.pt in the bucket, made by the p5_tokenize notebook or by the first notebook that needed it) is used when it exists."""
    rows, allrows = WANT[split], RAW[split]
    name = f"p5tok_{split}_{MAX_DOC_TOKENS}.pt"
    lab = LABEL_KEY if split != "test" else None
    docs_all = None
    if not QUICK_TEST:
        try:
            c = torch.load(fetch_model_file(name), map_location="cpu", weights_only=False)
            if c["ids_order"] == [r[ID_KEY] for r in allrows]:
                print(f"  {split}: tokenised cases loaded from the bucket cache ({name})")
                docs_all = [dict(id=i, ids=c["ids"][o:o2], labs=l, n=o2 - o) for i, l, o, o2 in zip(c["ids_order"], c["labs"], c["off"][:-1], c["off"][1:])]
        except Exception:
            pass
    if docs_all is None:
        full = len(rows) == len(allrows) and not QUICK_TEST
        src = allrows if full else rows
        print(f"  {split}: tokenising {len(src)} cases (no usable cache)")
        docs, st = encode_rows(bpe_tokenizer, src, ID_KEY, lab, MAX_DOC_TOKENS)
        print(f"  {split}: {st}")
        if full:
            off = np.concatenate([[0], np.cumsum([d["n"] for d in docs])]).astype(np.int64)
            torch.save(dict(ids=np.concatenate([d["ids"] for d in docs]), off=off, labs=[d["labs"] for d in docs], ids_order=[d["id"] for d in docs]), f"{WORK}/{name}")
            upload_model_file(f"{WORK}/{name}", name)
        docs_all = docs
    by = {d["id"]: d for d in docs_all}
    out = [by[r[ID_KEY]] for r in rows]
    for d in out:
        d["y"] = multi_hot(d["labs"], N_LABELS) if d["labs"] is not None else None
    return out

t0 = time.time()
TRAIN_DOCS, DEV_DOCS, TEST_DOCS = tokenised("train"), tokenised("dev"), tokenised("test")
assert [d["id"] for d in TEST_DOCS] == TEST_IDS
del RAW, WANT
Ytr, Ydv = np.stack([d["y"] for d in TRAIN_DOCS]), np.stack([d["y"] for d in DEV_DOCS])
for nm, ds, Y in (("train", TRAIN_DOCS, Ytr), ("dev", DEV_DOCS, Ydv)):
    print(f"{nm}: {len(ds)} cases, {Y.sum(1).mean():.2f} statutes per case, tokens/case median {int(np.median([d['n'] for d in ds]))}, max {max(d['n'] for d in ds)}; statute support min/median/max {int(Y.sum(0).min())}/{int(np.median(Y.sum(0)))}/{int(Y.sum(0).max())}")
print(f"test: {len(TEST_DOCS)} cases [{time.time() - t0:.0f}s]")
DEV_SEL_MASK = np.array([not DEV_IS_TEST_TEXT[d["id"]] for d in DEV_DOCS])
DEV_SEL_DOCS = [d for d, k in zip(DEV_DOCS, DEV_SEL_MASK) if k]
print(f"selection set: {len(DEV_SEL_DOCS)} dev cases")
assert len(DEV_SEL_DOCS) >= 4
if len(DEV_SEL_DOCS) < 1000: print("WARNING: the selection set is small, so epoch and threshold choices will be noisy")
SAN_DOCS = {d["id"]: {"ids": d["ids"][:1025].astype(np.int64).tolist()} for d in TRAIN_DOCS[:8]}

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

{'train': 42750, 'dev': 10181, 'test': 13019} | id: id | labels: labels
100 statutes (ids 0..99)
NOTE: 102 dev cases have the same text as a test case: they are excluded from every selection step (selection set = the other 10079 dev cases).


  train: tokenised cases loaded from the bucket cache (p5tok_train_16384.pt)


  dev: tokenised cases loaded from the bucket cache (p5tok_dev_16384.pt)


  test: tokenised cases loaded from the bucket cache (p5tok_test_16384.pt)
train: 42750 cases, 3.70 statutes per case, tokens/case median 580, max 16384; statute support min/median/max 272/679/11730
dev: 10181 cases, 3.92 statutes per case, tokens/case median 731, max 16384; statute support min/median/max 68/175/2936
test: 13019 cases [10s]
selection set: 10079 dev cases


## 10. Load the Phase-1 checkpoint, check it, define how a model is built

In [10]:
t0 = time.time()
_ck = torch.load(fetch_model_file(PHASE1_CHECKPOINT), map_location="cpu", mmap=True)
PHASE1_STATE = _ck["model"] if "model" in _ck else _ck
del _ck
_lm = build_phase1_lm()
PHASE1_PARAMS = sum(p.numel() for p in _lm.parameters())
SANITY_LOSS = lm_sanity_loss(_lm, SAN_DOCS, device)
print(f"Phase-1 model {MODEL_KEY}: {PHASE1_PARAMS:,} parameters; next-token loss on the case texts = {SANITY_LOSS:.3f} nats "
      f"(far below {math.log(VOCAB):.1f} = trained weights loaded; about {math.log(VOCAB):.1f} = random weights) [{time.time()-t0:.0f}s]")
assert SANITY_LOSS < 8.0, "the Phase-1 weights do not seem to be loaded correctly"
del _lm; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None


def new_backbone(lm):
    return DecoderBackbone(lm, bidirectional=False)


def build_model(C, log=print):
    """Phase-1 model -> backbone -> document model (chunk pooling + multi-label head) -> adaptation technique."""
    if C.get("RANDOM_INIT"):     # control: same architecture with freshly initialised weights (the Phase-1 weights are NOT loaded)
        _orig_lsd = nn.Module.load_state_dict
        nn.Module.load_state_dict = lambda self, *a, **k: None
        try:
            lm = build_phase1_lm()
        finally:
            nn.Module.load_state_dict = _orig_lsd
    else:
        lm = build_phase1_lm()
    bb = new_backbone(lm)
    model = DocModel(bb, bb.d_model, C)
    info = apply_technique(model, C, log)
    return model.to(device), info


def free():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

Phase-1 model moe_hc: 218,687,296 parameters; next-token loss on the case texts = 3.886 nats (far below 10.6 = trained weights loaded; about 10.6 = random weights) [9s]


## 11. Train every setup

In [11]:
import hashlib
RES_DIR = f"{WORK}/p5res_{SWEEP_NAME}"
os.makedirs(RES_DIR, exist_ok=True)
SWEEP_START = time.time()


def make_C(ov):
    C = dict(BASE, **ov)
    if "LR" not in ov:
        C["LR"] = LR_BY_TECH.get(C["TECHNIQUE"], C["LR"])
    return C


def cfg_key(C, n_train=None):
    c = {k: v for k, v in C.items()}
    return hashlib.md5(json.dumps([c, n_train or len(TRAIN_DOCS), len(DEV_DOCS), QUICK_TEST, MAX_DOC_TOKENS], sort_keys=True, default=str).encode()).hexdigest()[:10]


def cache_name(setup):
    return f"p5res_{SWEEP_NAME}__{setup}.pt"


def cache_load(setup, key):
    name = cache_name(setup)
    try:
        r = torch.load(fetch_model_file(name, f"{RES_DIR}/{name}"), map_location="cpu", weights_only=False)
        return r if r.get("key") == key else None
    except Exception:
        return None


def cache_save(setup, r):
    path = f"{RES_DIR}/{cache_name(setup)}"
    torch.save(r, path)
    upload_model_file(path, cache_name(setup))


def load_teacher(name):
    """Teacher scores {case id: (L,) logits} for the training cases: the teacher's own result file (made with SAVE_TRAIN_LOGITS=True). Waits if the notebook that makes it is still running."""
    t0 = time.time()
    while True:
        try:
            t = torch.load(fetch_model_file(name), map_location="cpu", weights_only=False); break
        except Exception as e:
            if (time.time() - t0) / 60 >= TEACHER_WAIT_MIN:
                raise FileNotFoundError(f"teacher file {name} not in the bucket after {TEACHER_WAIT_MIN} min ({e}); run the notebook that makes it first") from e
            print(f"  teacher file {name} not in the bucket yet; waiting ({(time.time() - t0) / 60:.0f} of {TEACHER_WAIT_MIN} min)"); time.sleep(120)
    assert t.get("train_logits") is not None, f"{name} holds no train_logits: the teacher setup needs SAVE_TRAIN_LOGITS=True"
    return t


def eval_scores(Y, P, sel, thr_modes=THR_MODES):
    """All the scores of a run, on the selection set: at 0.5, at the best global threshold, at per-label / shrunk thresholds (optimistic: fitted on the same cases) and the honest cross-fit estimate."""
    Ys, Ps = Y[sel], P[sel]
    out = dict(at05=ml_scores(Ys, apply_thr(Ps, 0.5)))
    thr = {m: fit_thresholds(Ys, Ps, m) for m in thr_modes}
    out["thr"] = thr
    out["tuned"] = {m: ml_scores(Ys, apply_thr(Ps, t)) for m, t in thr.items()}
    out["honest"], out["honest_sd"] = cross_fit(Ys, Ps, thr_modes, reps=CROSSFIT_REPS)
    out["best_mode"] = max(out["honest"], key=out["honest"].get)
    return out


def run_setup(name, ov, docs=None):
    docs = docs or TRAIN_DOCS
    C = make_C(ov); key = cfg_key(C, len(docs))
    r = cache_load(name, key)
    if r is not None:
        print(f"[{name}] cached (best epoch {r['best_epoch']}, selection-set macro-F1 {r['scores']['tuned']['global']['macro_F1']:.4f} at the best global threshold)")
        return r
    train = docs
    if C["KD_TEACHER"]:
        teacher = load_teacher(C["KD_TEACHER"])
        tl = teacher["train_logits"]
        tmap = dict(zip(tl["ids"], tl["logits"]))
        miss = [d["id"] for d in train if d["id"] not in tmap]
        assert not miss, f"{len(miss)} training cases have no teacher scores (the teacher must have been trained on at least the same cases)"
        train = [dict(d, kd=np.asarray(tmap[d["id"]], dtype=np.float32)) for d in docs]
        del teacher, tl, tmap
    print(f"== {name}: {ov}")
    model, info = build_model(C)
    ck_name = f"p5ck_{SWEEP_NAME}__{name}.pt"
    def ck_save(st):
        t_s = time.time(); st["key"] = key
        torch.save(st, f"{RES_DIR}/{ck_name}"); upload_model_file(f"{RES_DIR}/{ck_name}", ck_name)
        print(f"  checkpoint after epoch {st['epoch']} saved to the bucket ({os.path.getsize(f'{RES_DIR}/{ck_name}') / 1e6:.0f} MB, {time.time() - t_s:.0f}s)")
    def ck_load():
        try:
            st = torch.load(fetch_model_file(ck_name), map_location="cpu", weights_only=False)
            return st if st.get("key") == key else None
        except Exception:
            return None
    res = train_doc(model, train, DEV_SEL_DOCS, C, device, dev_sub=C["DEV_SUB"],
                    ckpt_save=ck_save if EPOCH_CHECKPOINTS else None, ckpt_load=ck_load if EPOCH_CHECKPOINTS else None)      # epochs are chosen on dev cases that are NOT test cases
    load_trainable_state(model, res["best_state"])
    if C["SAVE_WEIGHTS"]:
        wname = f"p5w_{SWEEP_NAME}__{name}.pt"
        torch.save(dict(setup=name, overrides=ov, C={k: v for k, v in C.items()}, state=res["best_state"], best_epoch=res["best_epoch"]), f"{RES_DIR}/{wname}")
        upload_model_file(f"{RES_DIR}/{wname}", wname)
        print(f"  trained parameters saved to the bucket as {wname} ({os.path.getsize(f'{RES_DIR}/{wname}') / 1e6:.0f} MB)")
    dev_lg, dinfo, attn = predict_logits(model, DEV_DOCS, C, device, collect_attn=True)
    test_lg, tinfo = predict_logits(model, TEST_DOCS, C, device)
    train_lg = None
    if C["SAVE_TRAIN_LOGITS"]:
        lg, _ = predict_logits(model, docs, C, device)
        train_lg = dict(ids=[d["id"] for d in docs], logits=lg.astype(np.float32))
    Y = np.stack([d["y"] for d in DEV_DOCS]); P = sigmoid_np(dev_lg)
    sc = eval_scores(Y, P, DEV_SEL_MASK)
    try:        # the same code as the course evaluator, on the selection set, with the best threshold mode (a check of our own scores; never stops a run)
        sel_ids = [i for i, k in zip([d["id"] for d in DEV_DOCS], DEV_SEL_MASK) if k]
        _off, _, _ = dev_codabench_check(sel_ids, Y[DEV_SEL_MASK], apply_thr(P[DEV_SEL_MASK], sc["thr"][sc["best_mode"]]), f"{RES_DIR}/dev_check", name)
        sc["official_dev"] = _off
        _ours = float(np.mean(sc["tuned"][sc["best_mode"]]["per_label_F1"][:_off["labels"]]))      # the evaluator takes the label count from the largest gold id (here all 100 statutes occur)
        if abs(_off["macro_F1"] - _ours) > 1e-6:
            print(f"  WARNING: official macro-F1 {_off['macro_F1']:.6f} differs from ours {_ours:.6f}")
    except Exception as e:
        print(f"  (official evaluator check skipped: {type(e).__name__}: {e})")
    r = dict(key=key, name=name, overrides=ov, C={k: v for k, v in C.items()}, info=info, history=res["history"], best_epoch=res["best_epoch"],
             dev_ids=[d["id"] for d in DEV_DOCS], dev_y=Y.astype(np.uint8), dev_sel_mask=DEV_SEL_MASK, dev_logits=dev_lg.astype(np.float32), test_ids=[d["id"] for d in TEST_DOCS], test_logits=test_lg.astype(np.float32),
             train_logits=train_lg, scores=sc, dev_attention=attn,
             train_seconds=res["train_seconds"], peak_mem_gb=res["peak_mem_gb"], train_tokens_per_sec=res["tokens_per_sec"],
             infer_dev=dinfo, infer_test=tinfo, n_train_docs=len(docs), phase1_params=PHASE1_PARAMS)
    cache_save(name, r)
    t = sc["tuned"]
    print(f"### {name}: selection-set macro-F1 {sc['at05']['macro_F1']:.4f} @0.5 | {t['global']['macro_F1']:.4f} global thr {sc['thr']['global'][0]:.2f} | per-label {t['per_label']['macro_F1']:.4f} | shrunk {t['shrunk']['macro_F1']:.4f} "
          f"| honest (cross-fit) {({k: round(v, 4) for k, v in sc['honest'].items()})} | micro-F1 {t['global']['micro_F1']:.4f} | subset acc {t['global']['subset_acc']:.4f} | best epoch {res['best_epoch']} | "
          f"train {res['train_seconds'] / 60:.1f} min, {res['tokens_per_sec']:.0f} tok/s, peak {res['peak_mem_gb']} GB | inference {dinfo['tokens_per_sec']:.0f} tok/s")
    del model, res; free()
    return r


RESULTS, SETUP_MIN = {}, []
for name, ov in SWEEP:
    elapsed = (time.time() - SWEEP_START) / 60
    need = 1.15 * max(SETUP_MIN) if SETUP_MIN else 0          # the longest setup so far, with a margin: do not START what cannot finish in time
    if elapsed + need > MAX_SWEEP_HOURS * 60 and cache_load(name, cfg_key(make_C(ov), len(TRAIN_DOCS))) is None:
        print(f"{elapsed:.0f} min used, a setup needs about {need:.0f} min, limit {MAX_SWEEP_HOURS} h: not starting {name} (Run All again in a new session to continue)"); continue
    t_s = time.time()
    RESULTS[name] = run_setup(name, ov)
    if time.time() - t_s > 60:
        SETUP_MIN.append((time.time() - t_s) / 60)
print(f"sweep done in {(time.time() - SWEEP_START) / 60:.0f} min")

== moe_lorma_long: {'TECHNIQUE': 'lorma', 'SAVE_WEIGHTS': True}
Technique lorma: trainable 4,285,797 of 192,213,093 parameters (1.9598% of the Phase-1 model's 218,687,296); stored size 769.9 MB
training: 42750 train documents, 6000 per epoch x 16 epochs, 375 optimizer steps/epoch, loss bce_pw, head bilstm; pos_weight range 0.23-1.43
  epoch 1: train_loss=0.96427, train_seconds=1648.1, dev_macroF1=0.0924, dev_macroF1_tuned=0.2488, tuned_thr=0.06, dev_microF1_tuned=0.3598, dev_subset_acc_tuned=0.001, eval_seconds=68.59


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p5ck_p5_moe_lorma__moe_lorma_long.pt to bucket kavincg23/cs781-models
  checkpoint after epoch 1 saved to the bucket (69 MB, 7s)
  epoch 2: train_loss=0.81538, train_seconds=3366.3, dev_macroF1=0.1453, dev_macroF1_tuned=0.2881, tuned_thr=0.08, dev_microF1_tuned=0.4095, dev_subset_acc_tuned=0.011, eval_seconds=69.42


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (59 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p5res_p5_moe_lorma__moe_lorma_long.pt to bucket kavincg23/cs781-models
### moe_lorma_long: selection-set macro-F1 0.2501 @0.5 | 0.3654 global thr 0.13 | per-label 0.3859 | shrunk 0.3770 | honest (cross-fit) {'global': 0.3637, 'per_label': 0.3708, 'shrunk': 0.3721} | micro-F1 0.4704 | subset acc 0.0389 | best epoch 14 | train 438.0 min, 4897 tok/s, peak 2.413 GB | inference 42369 tok/s
sweep done in 470 min


## 12. Comparison table

In [12]:
def _t(r, m="global"):
    return r["scores"]["tuned"][m]
rows = sorted(RESULTS.items(), key=lambda kv: -max(kv[1]["scores"]["honest"].values()))
print("All scores are on the SELECTION set (dev cases that are not test cases). macro-F1 is the primary metric. 'honest' = thresholds fitted on one half of the cases and scored on the other half (mean of repeated splits): the fair number. 'glob/pl/shr' = global / per-statute / shrunk thresholds fitted on the same cases (optimistic).")
print(f"{'setup':24s} {'F1@.5':>7s} {'glob':>7s} {'pl':>7s} {'shr':>7s} {'honest-g':>8s} {'honest-pl':>9s} {'honest-sh':>9s} {'mode':>9s} {'microF1':>7s} {'subsetAcc':>9s} {'P':>6s} {'R':>6s} {'ep':>3s} {'train M':>8s} {'%bb':>6s} {'MB':>7s} {'GB':>5s} {'min':>6s} {'tr tok/s':>9s} {'inf tok/s':>9s} {'inf doc/s':>9s}")
for n, r in rows:
    s, i, h = r["scores"], r["info"], r["scores"]["honest"]; g = _t(r)
    print(f"{n:24s} {s['at05']['macro_F1']:7.4f} {g['macro_F1']:7.4f} {_t(r, 'per_label')['macro_F1']:7.4f} {_t(r, 'shrunk')['macro_F1']:7.4f} {h['global']:8.4f} {h['per_label']:9.4f} {h['shrunk']:9.4f} {s['best_mode']:>9s} "
          f"{g['micro_F1']:7.4f} {g['subset_acc']:9.4f} {g['macro_P']:6.3f} {g['macro_R']:6.3f} {r['best_epoch']:3d} {i['params_trainable'] / 1e6:8.2f} {i['trainable_pct_of_backbone']:6.2f} {i['model_size_mb']:7.0f} {(r['peak_mem_gb'] or 0):5.1f} "
          f"{r['train_seconds'] / 60:6.1f} {r['train_tokens_per_sec']:9.0f} {r['infer_dev']['tokens_per_sec']:9.0f} {r['infer_dev']['docs_per_sec']:9.2f}")
if rows:
    n, r = rows[0]
    print(f"\nBest: {n}. per-epoch selection-sample macro-F1 at the best global threshold:", [h["dev_macroF1_tuned"] for h in r["history"]])

print("\nOfficial evaluator code (evaluate_phase5.py logic) on the selection set, each run at its best threshold mode (thresholds fitted on the same cases: optimistic; Codabench scores were about 0.015 below dev in Phase 4):")
for n, r in rows:
    o = r["scores"].get("official_dev")
    if o: print(f"  {n:24s} Macro-F1 {o['macro_F1']:.6f}  Micro-F1 {o['micro_F1']:.6f}  Subset Accuracy {o['subset_acc']:.6f}  ({o['examples']} examples, {o['labels']} labels)")

All scores are on the SELECTION set (dev cases that are not test cases). macro-F1 is the primary metric. 'honest' = thresholds fitted on one half of the cases and scored on the other half (mean of repeated splits): the fair number. 'glob/pl/shr' = global / per-statute / shrunk thresholds fitted on the same cases (optimistic).
setup                      F1@.5    glob      pl     shr honest-g honest-pl honest-sh      mode microF1 subsetAcc      P      R  ep  train M    %bb      MB    GB    min  tr tok/s inf tok/s inf doc/s
moe_lorma_long            0.2501  0.3654  0.3859  0.3770   0.3637    0.3708    0.3721    shrunk  0.4704    0.0389  0.333  0.433  14     4.29   1.96     770   2.4  438.0      4897     42369     28.58

Best: moe_lorma_long. per-epoch selection-sample macro-F1 at the best global threshold: [0.2488, 0.2881, 0.3117, 0.3166, 0.3358, 0.3392, 0.342, 0.3475, 0.3449, 0.354, 0.3541, 0.3587, 0.3661, 0.3688, 0.3664, 0.3676]

Official evaluator code (evaluate_phase5.py logic) on the

## 13. Submission files

In [13]:
# Submission files are written only for runs trained on ALL training cases (subset runs are for ranking only; their results stay cached in the bucket for the ensemble).
SUBMIT_TOP_K = 4          # at most this many runs get a zip (best first, by the honest cross-fit macro-F1 of their best threshold mode)
SUB_DIR = f"{WORK}/p5_submissions_{SWEEP_NAME}"
os.makedirs(SUB_DIR, exist_ok=True)
_full = sorted([(n, r) for n, r in RESULTS.items() if r["n_train_docs"] >= N_ALL_TRAIN], key=lambda kv: -max(kv[1]["scores"]["honest"].values()))[:SUBMIT_TOP_K]
if not _full:
    print("no run in this notebook was trained on all training cases: no submission file written (all results are cached in the bucket)")
for name, r in _full:
    mode = r["scores"]["best_mode"]; thr = r["scores"]["thr"][mode]
    preds = apply_thr(sigmoid_np(r["test_logits"]), thr)
    jpath, zpath = write_submission(r["test_ids"], preds, SUB_DIR, f"{SWEEP_NAME}_{name}_{mode}", TEST_IDS, N_LABELS)
    print(f"{os.path.basename(zpath):72s} {preds.sum(1).mean():.2f} statutes/case predicted (dev gold {r['scores']['tuned'][mode]['mean_true_labels']:.2f}); honest macro-F1 {r['scores']['honest'][mode]:.4f} ({mode} thresholds)")
    check_test_file(jpath, TEST_IDS, N_LABELS)                                   # the Codabench checks on the file we are about to submit
    print("   file check passed: one object per test case, only valid unique integer statute ids")
    ids_sel = [i for i, k in zip(r["dev_ids"], r["dev_sel_mask"]) if k]
    P_sel = sigmoid_np(r["dev_logits"])[np.asarray(r["dev_sel_mask"])]
    try:
        dev_codabench_check(ids_sel, np.asarray(r["dev_y"])[np.asarray(r["dev_sel_mask"])], apply_thr(P_sel, thr), SUB_DIR, f"{SWEEP_NAME}_{name}_{mode}")
    except Exception as ex:
        print(f"  (official evaluator check on dev skipped: {type(ex).__name__}: {ex})")
    upload_model_file(zpath)

print(f"dev_truth.json and *_dev_prediction.json are in {SUB_DIR}: `python evaluate_phase5.py --truth dev_truth.json --prediction <file>` gives the same three numbers as the lines above.")

submission_p5_moe_lorma_moe_lorma_long_shrunk.zip                        4.78 statutes/case predicted (dev gold 3.91); honest macro-F1 0.3721 (shrunk thresholds)
   file check passed: one object per test case, only valid unique integer statute ids
  [official evaluator] Examples: 10079  Labels: 100  Macro-F1: 0.376960  Micro-F1: 0.475752  Subset Accuracy: 0.040579


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded submission_p5_moe_lorma_moe_lorma_long_shrunk.zip to bucket kavincg23/cs781-models
dev_truth.json and *_dev_prediction.json are in /kaggle/working/p5_submissions_p5_moe_lorma: `python evaluate_phase5.py --truth dev_truth.json --prediction <file>` gives the same three numbers as the lines above.


## 14. Save the summary

In [14]:
SUMMARY = dict(sweep=SWEEP_NAME, model=MODEL_KEY, phase1_checkpoint=PHASE1_CHECKPOINT, phase1_params=PHASE1_PARAMS, sanity_loss=SANITY_LOSS,
               full_run=FULL_RUN, n_train=len(TRAIN_DOCS), n_dev=len(DEV_DOCS), n_test=len(TEST_DOCS), base=BASE,
               results={n: {k: v for k, v in r.items() if k not in ("dev_logits", "test_logits", "train_logits", "dev_attention", "dev_ids", "test_ids", "dev_sel_mask", "dev_y")} for n, r in RESULTS.items()},
               total_minutes=round((time.time() - NB_START) / 60, 1))
path = f"{WORK}/summary_{SWEEP_NAME}.json"
with open(path, "w") as f:
    json.dump(SUMMARY, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
upload_model_file(path)
for p in sorted(os.listdir(SUB_DIR)):
    print("  ", f"{SUB_DIR}/{p}", os.path.getsize(f"{SUB_DIR}/{p}") // 1024, "KB")
print(f"done in {(time.time()-NB_START)/60:.0f} min. Send back: summary_{SWEEP_NAME}.json and the zips you want to submit.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded summary_p5_moe_lorma.json to bucket kavincg23/cs781-models
   /kaggle/working/p5_submissions_p5_moe_lorma/dev_truth.json 444 KB
   /kaggle/working/p5_submissions_p5_moe_lorma/p5_moe_lorma_moe_lorma_long_shrunk_dev_prediction.json 479 KB
   /kaggle/working/p5_submissions_p5_moe_lorma/p5_moe_lorma_moe_lorma_long_shrunk_prediction.json 616 KB
   /kaggle/working/p5_submissions_p5_moe_lorma/submission_p5_moe_lorma_moe_lorma_long_shrunk.zip 113 KB
done in 471 min. Send back: summary_p5_moe_lorma.json and the zips you want to submit.
